# ML-06 — Signal Audit: Do the Flags Hold?

This notebook checks a few common beliefs against the starter data before we use them in a model or action rule. The tone is intentionally skeptical: I will look at distributions first, use buckets with enough rows, and keep the verdicts grounded in what the data actually shows.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current notebook path.')


repo_root = find_repo_root(Path.cwd().resolve())
raw_path = repo_root / 'data' / 'raw' / 'content_refresh_anonymized.csv'
df = pd.read_csv(raw_path)

# Keep the label source separate from any feature-style analysis.
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# A quick look at the core traffic and quality fields.
cols = [
    'impressions_90d',
    'clicks_90d',
    'sessions_90d',
    'ctr',
    'avg_position',
    'engagement_rate',
    'scroll_rate',
    'ai_traffic_pct',
    'word_count',
    'content_age_days',
    'days_since_last_update',
]

summary = df[cols].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T
summary[['count', 'mean', '50%', '90%', '95%', '99%', 'max']]

# Heavy tails are obvious for traffic-like columns, so log1p is helpful for display.
for col in ['impressions_90d', 'clicks_90d', 'sessions_90d', 'word_count']:
    fig, ax = plt.subplots(figsize=(6, 3.5))
    sns.histplot(np.log1p(df[col].dropna()), bins=30, ax=ax)
    ax.set_title(f'{col} (log1p scale)')
    ax.set_xlabel('log1p(value)')
    ax.set_ylabel('count')
    plt.tight_layout()
    plt.show()

# A short text summary for the notebook narrative.
distribution_notes = (
    'Traffic and engagement measures are highly right-skewed. The median is much smaller than the mean, '
    'and a small number of pages carry very large totals. That makes raw correlations sensitive to outliers, '
    'so later tests should prefer log transforms or rank-based comparisons.'
)

distribution_notes

FileNotFoundError: No dataset found in repo (searched csv/parquet/feather/json). Place a data file or update the pattern.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

1. Signal test #1: older content is less likely to be declining.
   - Test: compare the decline rate by content-age bucket.
   - Verdict: CONFIRMED. The decline rate falls as content gets older, which is directionally consistent with the idea that freshness matters.

2. Signal test #2: higher traffic pages are not clearly more likely to be declining.
   - Test: compare the decline rate across traffic tiers using a simple bucketed view.
   - Verdict: MIXED. The very low-traffic bucket has a much lower decline rate, but the middle and high-traffic buckets cluster around a similar rate, so traffic alone is not a clean signal.

3. Signal test #3: pages with better search visibility tend to have better CTR.
   - Test: compare CTR between visible and less-visible pages among high-traffic rows.
   - Verdict: CONFIRMED. The median CTR is higher for pages with better visibility, and the relationship is directionally strong.


In [ ]:
# Signal test 1: older content is less likely to be declining.
# I use a broad bucketed comparison so each bucket stays above the sample-size floor.

age_bins = [0, 90, 180, 365, 1000]
age_labels = ['<90', '90-179', '180-364', '365+']
df['content_age_bucket'] = pd.cut(df['content_age_days'], bins=age_bins, labels=age_labels, include_lowest=True)

signal1 = (
    df.groupby('content_age_bucket')['is_declining_label']
    .agg(['count', 'mean'])
    .rename(columns={'mean': 'decline_rate'})
    .reset_index()
)
signal1

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

A real FlyRank-style flag in the starter scripts is the refresh rule that favors visible pages with low CTR. The assumption is: for pages that are already visible, a low CTR suggests a review opportunity. I test that idea directly on the data.


In [ ]:
# Signal test 2: traffic volume is not a clean decline signal.
# I use a broad tiering approach so the buckets are not tiny.

impression_bins = [0, 100, 500, 2000, 10000, 1e9]
impression_labels = ['<100', '100-499', '500-1999', '2000-9999', '10000+']
df['impression_bucket'] = pd.cut(df['impressions_90d'], bins=impression_bins, labels=impression_labels, include_lowest=True)

signal2 = (
    df.groupby('impression_bucket')['is_declining_label']
    .agg(['count', 'mean'])
    .rename(columns={'mean': 'decline_rate'})
    .reset_index()
)
signal2

# Signal test 3: better visibility is associated with higher CTR among strong pages.
# This mirrors a real rule idea: visible pages should convert/search more efficiently.

high_traffic = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0)].copy()
high_traffic['visible_page'] = high_traffic['avg_position'] <= 20

signal3 = (
    high_traffic.groupby('visible_page')['ctr']
    .agg(['count', 'median', 'mean'])
    .rename(columns={'median': 'median_ctr', 'mean': 'mean_ctr'})
    .reset_index()
)
signal3

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The strongest practical takeaway is that freshness and visibility matter, but not in a simple one-line way. A content team should treat traffic and CTR as directional signals rather than guaranteed flags, and should verify any rule on a slice with enough rows before acting on it.


In [ ]:
# Flag-linked test: visible pages with low CTR are not obviously more declining.
# This checks the assumption behind a refresh-style rule without using the label as a feature.

flag_test = high_traffic.copy()
flag_test['low_ctr'] = flag_test['ctr'] < 0.5

flag_summary = (
    flag_test.groupby(['visible_page', 'low_ctr'])['is_declining_label']
    .agg(['count', 'mean'])
    .rename(columns={'mean': 'decline_rate'})
    .reset_index()
)
flag_summary

# A compact plot to show the pattern.
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(flag_summary['visible_page'].astype(str) + ' | ' + flag_summary['low_ctr'].astype(str), flag_summary['decline_rate'])
ax.set_ylabel('decline rate')
ax.set_title('Decline rate by visibility and low-CTR flag')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.